In [47]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
import json
import os

In [55]:
data= pd.read_csv('/Users/prathikkundaragi/Downloads/fraud_dataset.csv', low_memory=True)
data.head(10)

,step,type,amount,nameOrig,oldbalanceOrg,newbalanceOrig,nameDest,oldbalanceDest,newbalanceDest,isFraud,isFlaggedFraud
0,1,PAYMENT,9839.64,C1231006815,170136.00,160296.36,M1979787155,0.0,0.00,0,0
1,1,PAYMENT,1864.28,C1666544295,21249.00,19384.72,M2044282225,0.0,0.00,0,0
2,1,TRANSFER,181.00,C1305486145,181.00,0.00,C553264065,0.0,0.00,1,0
3,1,CASH_OUT,181.00,C840083671,181.00,0.00,C38997010,21182.0,0.00,1,0
4,1,PAYMENT,11668.14,C2048537720,41554.00,29885.86,M1230701703,0.0,0.00,0,0
5,1,PAYMENT,7817.71,C90045638,53860.00,46042.29,M573487274,0.0,0.00,0,0
6,1,PAYMENT,7107.77,C154988899,183195.00,176087.23,M408069119,0.0,0.00,0,0
7,1,PAYMENT,7861.64,C1912850431,176087.23,168225.59,M633326333,0.0,0.00,0,0
8,1,PAYMENT,4024.36,C1265012928,2671.00,0.00,M1176932104,0.0,0.00,0,0
9,1,DEBIT,5337.77,C712410124,41720.00,36382.23,C195600860,41898.0,40348.79,0,0


In [56]:
data.drop(['isFlaggedFraud','nameOrig','step','nameDest'], axis=1, inplace=True)

In [57]:
data.shape

(6362620, 7)

In [58]:
# One-hot encode the 'type' column
data = pd.get_dummies(data, columns=['type'], drop_first=False, dtype=int)
data.head()

,amount,oldbalanceOrg,newbalanceOrig,oldbalanceDest,newbalanceDest,isFraud,type_CASH_IN,type_CASH_OUT,type_DEBIT,type_PAYMENT,type_TRANSFER
0,9839.64,170136.0,160296.36,0.0,0.0,0,0,0,0,1,0
1,1864.28,21249.0,19384.72,0.0,0.0,0,0,0,0,1,0
2,181.00,181.0,0.00,0.0,0.0,1,0,0,0,0,1
3,181.00,181.0,0.00,21182.0,0.0,1,0,1,0,0,0
4,11668.14,41554.0,29885.86,0.0,0.0,0,0,0,0,1,0


In [59]:
# Save the full dataset as parquet file
data.to_parquet("/Users/prathikkundaragi/MY PROJECTS/Fraud Dataset/model_data_full.parquet")

In [5]:
data.isna().sum()

type              0
amount            0
oldbalanceOrg     0
newbalanceOrig    0
oldbalanceDest    0
newbalanceDest    0
isFraud           0
dtype: int64

In [6]:
data['type'].value_counts().to_frame()

,count
type,
CASH_OUT,2237500
PAYMENT,2151495
CASH_IN,1399284
TRANSFER,532909
DEBIT,41432


In [7]:
def split_data(data, target_col, split_ratio=0.8):
    train, test = train_test_split(data, test_size=1-split_ratio, random_state=42, stratify=data[target_col])
    train_X = train.drop(target_col, axis=1)
    train_y = train[[target_col]]
    test_X = test.drop(target_col, axis=1)
    test_y = test[[target_col]]
    return train_X, test_X, train_y, test_y

In [9]:
train_X, train_y, test_X, test_y =split_data(data ,'isFraud')

In [11]:

#getting segment wise data
segment_data={}
for segment in data['type'].unique():
    segment_data[segment] = {}
    segment_data[segment]['data'] = data[data['type']==segment]
    segment_data[segment]['train_X'], segment_data[segment]['test_X'], segment_data[segment]['train_y'], segment_data[segment]['test_y'] = split_data(segment_data[segment]['data'], 'isFraud') 


In [ ]:
#saving data dictioanryn to access in model files
data_dict= {'segment':list(segment_data.keys()),'data': list(segment_data[list(segment_data.keys())[0]].keys() ) }
with open('/Users/prathikkundaragi/Downloads/fraud_segment_data_dict.json', 'w') as f:
    json.dump(data_dict, f)

In [51]:
for segment,subdata in segment_data.items(): 
   for key, value in subdata.items():
        data_dict={}
        print(f"Segment: {segment}, Key: {key}")
        value.to_parquet(f"/Users/prathikkundaragi/MY PROJECTS/Fraud Dataset/fraud_{segment}_{key}.parquet")


Segment: PAYMENT, Key: data
Segment: PAYMENT, Key: train_X
Segment: PAYMENT, Key: test_X
Segment: PAYMENT, Key: train_y
Segment: PAYMENT, Key: test_y
Segment: TRANSFER, Key: data
Segment: TRANSFER, Key: train_X
Segment: TRANSFER, Key: test_X
Segment: TRANSFER, Key: train_y
Segment: TRANSFER, Key: test_y
Segment: CASH_OUT, Key: data
Segment: CASH_OUT, Key: train_X
Segment: CASH_OUT, Key: test_X
Segment: CASH_OUT, Key: train_y
Segment: CASH_OUT, Key: test_y
Segment: DEBIT, Key: data
Segment: DEBIT, Key: train_X
Segment: DEBIT, Key: test_X
Segment: DEBIT, Key: train_y
Segment: DEBIT, Key: test_y
Segment: CASH_IN, Key: data
Segment: CASH_IN, Key: train_X
Segment: CASH_IN, Key: test_X
Segment: CASH_IN, Key: train_y
Segment: CASH_IN, Key: test_y


In [15]:
# read a parquet file saved earlier (uses existing `segment` and `key` variables)
path = f"/Users/prathikkundaragi/Downloads/fraud_{segment}_{key}.parquet"

try:
    df_parquet = pd.read_parquet(path)
    print(f"Loaded from file: {path}")
except Exception as e:
    print(f"Failed to read file: {e}\nFalling back to in-memory `parquet_data` dict (if available).")
    df_parquet = parquet_data.get(f"{segment}_{key}")
    if df_parquet is None:
        raise FileNotFoundError(f"No parquet file at {path} and no in-memory entry for key '{segment}_{key}'")

# show a quick preview
df_parquet.head()

Loaded from file: /Users/prathikkundaragi/Downloads/fraud_CASH_IN_test_y.parquet


,isFraud
4947743,0
1585963,0
2126007,0
5444700,0
5476353,0
